# 🚀 Image Classification Training - Optimized Version

This notebook provides an optimized training pipeline with:
- **Mixed Precision Training (AMP)**: 2-3x faster training
- **OneCycleLR Scheduler**: Better convergence
- **Mixup Augmentation**: Improved generalization
- **Gradient Clipping**: Training stability
- **Strong Data Augmentation**: Better accuracy

---

## 📦 1. Import Libraries and Setup

In [ ]:
import os
import sys
import time
from pathlib import Path
import json

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import autocast, GradScaler

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Import local modules
from dataset_loader import create_dataloaders
from model import create_model
from config import DATA_CONFIG, MODEL_CONFIG, TRAIN_CONFIG, OUTPUT_CONFIG
from train import Trainer, print_training_info

# Enable inline plotting
%matplotlib inline

# Print versions
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## ⚙️ 2. Configuration

Adjust these settings based on your needs:

In [ ]:
# Merge all configs
config = {
    **DATA_CONFIG,
    **MODEL_CONFIG,
    **TRAIN_CONFIG,
    **OUTPUT_CONFIG
}

# ========================================================================
# CUSTOMIZE YOUR SETTINGS HERE:
# ========================================================================

config['data_root'] = 'split_dataset'  # Dataset path
config['output_dir'] = 'outputs'       # Output directory

# Training hyperparameters (adjust as needed)
config['epochs'] = 30                  # Number of epochs
config['batch_size'] = 64              # Batch size (reduce if OOM)
config['learning_rate'] = 3e-4         # Base learning rate
config['max_lr'] = 3e-3                # Max learning rate for OneCycleLR

# Performance optimizations
config['use_amp'] = True               # Mixed precision training (faster)
config['use_mixup'] = True             # Mixup augmentation (better accuracy)
config['num_workers'] = 4              # Data loading workers (0 for debugging)

# Model selection
config['model_name'] = 'vit_tiny_patch16_224'  # Options: vit_tiny_patch16_224, vit_small_patch16_224, vit_base_patch16_224

# ========================================================================

# Print configuration
print("\n📋 Training Configuration:")
print("="*60)
for key, value in config.items():
    print(f"  {key:25s}: {value}")
print("="*60)

## 📊 3. Load Dataset

In [ ]:
# Create dataloaders
train_loader, val_loader, test_loader, num_classes, class_names = create_dataloaders(
    config['data_root'],
    config['batch_size'],
    config['num_workers'],
    config['image_size'],
    config.get('crop_size', None),
    config.get('pin_memory', True),
    config.get('persistent_workers', True),
    config.get('prefetch_factor', 2)
)

# Save class names
output_dir = Path(config['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / 'class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

print(f"\n✅ Dataset loaded successfully!")
print(f"📁 Classes: {class_names}")

### 🔍 Visualize Sample Images

In [ ]:
# Visualize a batch of training images
def denormalize(tensor):
    """Denormalize image tensor for visualization"""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

# Get a batch
images, labels = next(iter(train_loader))

# Plot 8 images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, ax in enumerate(axes.flat):
    if idx < len(images):
        img = denormalize(images[idx]).permute(1, 2, 0).numpy().clip(0, 1)
        ax.imshow(img)
        ax.set_title(f"Class: {class_names[labels[idx]]}")
        ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

## 🤖 4. Create Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Create model
model = create_model(
    num_classes,
    config['model_name'],
    config['pretrained'],
    config['freeze_backbone'],
    config.get('dropout', 0.1)
)

# Try PyTorch 2.0+ compile (optional speedup)
if config.get('use_compile', False) and hasattr(torch, 'compile'):
    try:
        print("🔥 Compiling model with PyTorch 2.0+ (this may take a minute)...")
        model = torch.compile(model, mode='max-autotune')
        print("✅ Model compiled successfully!\n")
    except Exception as e:
        print(f"⚠️ Model compilation failed: {e}\n")

# Print model info
print_training_info(config, device, train_loader, val_loader, num_classes, model)

## 🏋️ 5. Train Model

This cell will train the model. Progress will be shown with real-time updates.

In [ ]:
# Create trainer
trainer = Trainer(model, train_loader, val_loader, device, config)

# Start training
print("\n" + "="*80)
print(" " * 30 + "🚀 STARTING TRAINING")
print("="*80 + "\n")

trainer.train()

print("\n" + "="*80)
print(" " * 30 + "✅ TRAINING COMPLETE")
print("="*80)

## 📈 6. Visualize Training Results

### Accuracy Plot

In [ ]:
# Plot accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
epochs = range(1, len(trainer.history['train_acc']) + 1)
plt.plot(epochs, [acc * 100 for acc in trainer.history['train_acc']], 'b-', label='Train', linewidth=2)
plt.plot(epochs, [acc * 100 for acc in trainer.history['val_acc']], 'r-', label='Validation', linewidth=2)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Training vs Validation Accuracy', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs, trainer.history['train_loss'], 'b-', label='Train', linewidth=2)
plt.plot(epochs, trainer.history['val_loss'], 'r-', label='Validation', linewidth=2)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training vs Validation Loss', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Learning Rate Schedule

In [ ]:
# Plot learning rate
plt.figure(figsize=(10, 4))
epochs = range(1, len(trainer.history['learning_rates']) + 1)
plt.plot(epochs, trainer.history['learning_rates'], 'g-', linewidth=2)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.title('Learning Rate Schedule (OneCycleLR)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

### Metrics Summary

In [ ]:
# Plot metrics
plt.figure(figsize=(10, 5))
epochs = range(1, len(trainer.history['precision']) + 1)
plt.plot(epochs, trainer.history['precision'], 'g-', label='Precision', linewidth=2)
plt.plot(epochs, trainer.history['recall'], 'b-', label='Recall', linewidth=2)
plt.plot(epochs, trainer.history['f1'], 'r-', label='F1 Score', linewidth=2)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Precision, Recall, and F1 Score', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print final metrics
print("\n📊 Final Validation Metrics:")
print("="*60)
print(f"  Accuracy:  {trainer.history['val_acc'][-1]*100:.2f}%")
print(f"  Precision: {trainer.history['precision'][-1]:.4f}")
print(f"  Recall:    {trainer.history['recall'][-1]:.4f}")
print(f"  F1 Score:  {trainer.history['f1'][-1]:.4f}")
print("="*60)

## 🎯 7. Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(output_dir / 'best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Evaluate on test set
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Calculate metrics
test_acc = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
test_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
test_f1 = f1_score(all_labels, all_preds, average='macro')

print("\n🎯 Test Set Results:")
print("="*60)
print(f"  Accuracy:  {test_acc*100:.2f}%")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1 Score:  {test_f1:.4f}")
print("="*60)

### Confusion Matrix

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 💾 8. Save Results

All results are automatically saved to the output directory:
- Model checkpoint: `best_model.pth`
- Class names: `class_names.json`
- Training metrics: `training_metrics.json`
- Plots: Various PNG files

In [ ]:
print(f"\n📁 All results saved to: {output_dir}")
print(f"\n📦 Files saved:")
for file in output_dir.glob('*'):
    if file.is_file():
        size = file.stat().st_size / 1024 / 1024  # MB
        print(f"  - {file.name:40s} ({size:.2f} MB)")

## 🎉 Training Complete!

### Next Steps:
1. Check the saved model in `outputs/best_model.pth`
2. Use `inference.py` to make predictions on new images
3. Analyze the confusion matrix for misclassifications
4. Experiment with different hyperparameters

### Tips for Better Results:
- **Increase epochs** if validation accuracy is still improving
- **Adjust batch size** based on GPU memory
- **Try different models**: `vit_small_patch16_224` or `vit_base_patch16_224`
- **Enable/disable augmentations** based on your dataset
- **Reduce num_workers to 0** if you encounter data loading issues